# Review trade-offs, scenario losses and monitoring

Quantile bands express a capacity assumption. Slider exploration must not tune reported policy on test. LGD/EAD scenarios are unobserved assumptions in NT dollars. Monitoring is random-partition/synthetic, not actual temporal drift.

In [1]:
from pathlib import Path
import sys,json
import pandas as pd
from IPython.display import display,Image
ROOT=Path.cwd()
sys.path.insert(0,str(ROOT))
T=ROOT/'artifacts/tables'; F=ROOT/'artifacts/figures'
summary=json.loads((T/'run_summary.json').read_text())


In [2]:
display(pd.read_csv(T/'threshold_tradeoffs.csv'))
display(pd.read_csv(T/'expected_loss_scenarios.csv'))
display(pd.read_csv(T/'monitoring.csv'))
display(pd.read_csv(T/'monitoring_outcomes.csv'))
import duckdb
con=duckdb.connect(str(T/'portfolio.duckdb'),read_only=True)
for path in sorted((ROOT/'sql').glob('*.sql')):
    print(path.name)
    display(con.execute(path.read_text()).df())
con.close()

,threshold,simulated_acceptance_rate,review_rate,accepted_event_rate,event_capture,false_positive_rate,false_negative_rate,mean_accepted_score,tp,fp,fn,tn
0,0.056283,0.097338,0.902662,0.049573,0.977930,0.881601,0.022070,0.045494,1285,4140,29,556
1,0.066211,0.148087,0.851913,0.051685,0.964992,0.820273,0.035008,0.050859,1268,3852,46,844
2,0.076710,0.199667,0.800333,0.051667,0.952816,0.757666,0.047184,0.056175,1252,3558,62,1138
3,0.088073,0.245757,0.754243,0.054164,0.939117,0.702513,0.060883,0.061079,1234,3299,80,1397
4,0.100965,0.296839,0.703161,0.063341,0.914003,0.644165,0.085997,0.066815,1201,3025,113,1671
5,0.113433,0.347587,0.652413,0.070369,0.888128,0.586457,0.111872,0.072780,1167,2754,147,1942
6,0.124644,0.394343,0.605657,0.078059,0.859209,0.534710,0.140791,0.078282,1129,2511,185,2185
7,0.135210,0.444592,0.555408,0.087201,0.822679,0.480622,0.177321,0.084113,1081,2257,233,2439
8,0.147078,0.500166,0.499834,0.095476,0.781583,0.420997,0.218417,0.090430,1027,1977,287,2719
9,0.161226,0.552579,0.447421,0.101174,0.744292,0.364353,0.255708,0.096414,978,1711,336,2985


,probability,validation_score_quantile,assumed_lgd,assumed_ead_ntd,scenario_loss_ntd
0,0.088073,0.25,0.25,50000,1100.916754
1,0.088073,0.25,0.25,100000,2201.833508
2,0.088073,0.25,0.25,200000,4403.667017
3,0.088073,0.25,0.45,50000,1981.650157
4,0.088073,0.25,0.45,100000,3963.300315
5,0.088073,0.25,0.45,200000,7926.600630
6,0.088073,0.25,0.65,50000,2862.383561
7,0.088073,0.25,0.65,100000,5724.767122
8,0.088073,0.25,0.65,200000,11449.534243
9,0.147078,0.50,0.25,50000,1838.470726


,feature,comparison,psi,train_missing,test_missing
0,LIMIT_BAL,random train versus test,0.003458,0.0,0.0
1,utilization_latest,random train versus test,0.001779,0.0,0.0
2,payment_to_limit,random train versus test,0.006033,0.0,0.0
3,months_delayed,random train versus test,0.000636,0.0,0.0
4,score,random train versus test,0.002620,0.0,0.0
5,LIMIT_BAL,synthetic 1.5x limit shift versus test,0.726761,0.0,0.0


,split,n,event_rate,mean_score,brier,missing_cells
0,test,6010,0.218636,0.219318,0.132508,0
1,validation,6015,0.225935,0.222694,0.140055,0
2,calibration,6049,0.218549,0.218549,0.134157,0
3,train,11926,0.221449,0.222504,0.124076,0


01_portfolio_overview.sql


...

,split,SEX,clients,event_rate,mean_score,brier
0,calibration,1,2392,0.238712,0.234838,0.142692
1,calibration,2,3657,0.205360,0.207894,0.128574
2,test,1,2346,0.232737,0.229012,0.139162
3,test,2,3664,0.209607,0.213110,0.128247
4,train,1,4753,0.241111,0.233993,0.132332
5,train,2,7173,0.208420,0.214890,0.118606
6,validation,1,2397,0.254485,0.233883,0.153581
7,validation,2,3618,0.207020,0.215281,0.131093


04_risk_deciles.sql


,risk_decile,clients,events,event_rate,mean_score
0,1,601,425.0,0.707155,0.685817
1,2,601,261.0,0.434276,0.414753
2,3,601,148.0,0.246256,0.278218
3,4,601,94.0,0.156406,0.201233
4,5,601,99.0,0.164725,0.161098
5,6,601,93.0,0.154742,0.136275
6,7,601,80.0,0.133111,0.114243
7,8,601,52.0,0.086522,0.089121
8,9,601,33.0,0.054908,0.066631
9,10,601,29.0,0.048253,0.045787


05_high_risk_exposure.sql


,risk_band,clients,observed_credit_limit_ntd,scenario_loss_ntd
0,HIGH,893,122270000.0,1.585787e+07
1,VERY HIGH,554,53090000.0,1.736121e+07
